# 第69课 · CTC（Connectionist Temporal Classification，CTC） 前向算法（forward algorithm） — 所有路径概率求和，O(T·S) 纯 NumPy 实现

**学习目标**
- 理解 CTC 的核心挑战：不知道字符在哪一帧对齐（alignment）
- 掌握前向变量（forward variable，alpha） α 的递推定义
- 用纯 NumPy 实现 CTC 前向算法并数值验证

← **上一课**　[L68 · CTC 对齐原理](L68_ctc_alignment.ipynb)

> 上节课学习了 **CTC 对齐原理**：blank 符号、单调路径与标签折叠。  
> 本课将探讨 **CTC 前向算法**。

## 本课剧情：为什么朴素的路径枚举会"爆炸"？

上节课学了 CTC 的直觉：blank 符号让一个目标序列对应指数级多条路径。

但问题来了：**若要精确计算 P("ab" | 6帧)，我们需要枚举 3⁶=729 条路径，逐一压缩验证，再相加**。T=100帧时就是 3¹⁰⁰条路径——彻底不可行。

**前向算法（Forward Algorithm）**的解法：用动态规划，把"枚举→求和"变成"递推→合并"。

**关键设计**：把目标序列 `["a","b"]` 扩展为包含 blank 的序列 `l' = [∅, a, ∅, b, ∅]`（每个字符前后插 blank），长度 `S = 2L+1`。

**前向变量** α[t][s] = "在第 t 帧结束时，以 l'[s] 结束的所有合法路径的概率和"。

**递推关系（三种情况）**：
```
情况1：l'[s] = blank，或与前2步字符相同
   α[t][s] = (α[t-1][s] + α[t-1][s-1]) × P(l'[s] | t)

情况2：l'[s] = 字符，且 l'[s] ≠ l'[s-2]
   α[t][s] = (α[t-1][s] + α[t-1][s-1] + α[t-1][s-2]) × P(l'[s] | t)
```

最终：`log P(label | logits) = logsumexp(α[T-1][S-1], α[T-1][S-2])`

复杂度从指数 → **O(T·S)**，其中 `S = 2L + 1`；训练 ASR 成为可能。

In [ ]:
import numpy as np

In [ ]:
# 构造玩具示例：6 帧，词汇 {a:0, b:1, blank:2}，目标 'ab'
BLANK = 2
VOCAB_SIZE = 3
T = 6
LABELS = [0, 1]   # 'a', 'b'

np.random.seed(42)
logits = np.random.randn(T, VOCAB_SIZE)
# 转为 log 概率
# 数值稳定 log-softmax：先减最大值，避免 exp 溢出
logits_s = logits - logits.max(axis=1, keepdims=True)
log_probs = logits_s - np.log(np.exp(logits_s).sum(axis=1, keepdims=True))
print(f'log_probs shape: {log_probs.shape}  (T={T}, vocab={VOCAB_SIZE})')

## ✏️ 实现 CTC 前向算法

**签名**：
```python
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    # log_probs: (T, V) 对数概率矩阵
    # labels: list[int] 目标序列（不含 blank）
    # 返回: float  log P(labels | log_probs)
```

**5步实现路线**：

| 步骤 | 操作 | 关键点 |
|---|---|---|
| 1 | 构造扩展标签 `l'` | `[blank, L[0], blank, L[1], ..., blank]`，长度 `2L+1` |
| 2 | 初始化 α：`-inf`，设 `α[0][0]` 和 `α[0][1]` | 第0帧只能是 blank 或第1个字符 |
| 3 | 对 t=1..T-1：对 s=0..S-1 递推 | 情况1（blank/重复）取2项，情况2取3项 |
| 4 | 用 log-space 加法避免下溢 | `logsumexp(a, b) = a + log(1 + exp(b-a))` |
| 5 | 返回 `logsumexp(α[T-1][S-1], α[T-1][S-2])` | 两种结尾方式的概率和 |

**验收标准**：
- 与暴力枚举（`ctc_forward_brute_force`）结果误差 < 1e-5
- `T=6, L=["a","b"]` 时返回有限 float（不是 -inf）

**实现前先记一句**：`S = 2L + 1`，本课复杂度是 `O(T·S)`，不是 `O(T×2L)`。


### Log 域等价（实现用此式）

```text
log α[t, s] = log p[t, l'[s]] + logaddexp(
    log α[t-1, s],      # 停留
    log α[t-1, s-1],    # 跳一步
    log α[t-1, s-2]     # 跳两步（仅当 l'[s] ≠ l'[s-2]）
)
```

`np.logaddexp(a, b)` = `log(exp(a) + exp(b))`，数值稳定；三路前驱可用两次 `logaddexp` 叠出来。

In [ ]:
a, b = np.log([0.3, 0.7])
assert np.isclose(np.logaddexp(a, b), np.log(1.0))
print("✅ logaddexp 演示：log(exp(a)+exp(b)) 在 log 域里能稳定相加")

In [ ]:
def log_softmax(logits, axis=-1):
    m = logits.max(axis=axis, keepdims=True)
    shifted = logits - m
    return shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))

In [ ]:
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """CTC 前向算法：计算 log P(labels | log_probs)。

    Args:
        log_probs: (T, vocab_size) log-probability matrix.
        labels:    list of integer label ids (without blanks).
        blank:     blank token index.
                   注意：本笔记本 BLANK=2，请显式传入 blank=BLANK；
                   函数默认值 0 是 CTC 论文惯例，不适用于本例。

    Returns:
        log probability of the label sequence under CTC.
    """
    T, V = log_probs.shape
    L = len(labels)

    # 扩展标签：blank + label + blank + label + ... + blank
    lprime = [blank]
    for c in labels:
        lprime.append(c)
        lprime.append(blank)
    S = len(lprime)   # = 2L + 1

    NEG_INF = -1e30
    alpha = np.full((T, S), NEG_INF)

    # ✏️ TODO 步骤1：初始条件
    # 第 0 帧只能从扩展标签的前两个位置出发（blank 或第一个字符）
    # alpha[0, 0] = log_probs[0, lprime[0]]
    # if S > 1: alpha[0, 1] = log_probs[0, lprime[1]]

    # ✏️ TODO 步骤2：t=1..T-1 的递推（三个合法前驱）
    # for t in range(1, T):
    #     for s in range(S):
    #         val = alpha[t-1, s]                                   # 停留
    #         if s > 0:
    #             val = np.logaddexp(val, alpha[t-1, s-1])          # 跳1
    #         if s > 1 and lprime[s] != lprime[s-2]:                # 跳2（仅字符位）
    #             val = np.logaddexp(val, alpha[t-1, s-2])
    #         alpha[t, s] = val + log_probs[t, lprime[s]]

    # ✏️ TODO 步骤3：返回终态 log-sum
    # 最后一帧可停在位置 S-1（字符）或 S-2（blank），取 logaddexp
    # return np.logaddexp(alpha[-1, -1], alpha[-1, -2])

    raise NotImplementedError  # 完成步骤1-3后删除此行

## 半步练习 B · ctc_forward_brute_force（先写 / 先读此函数）

先把“求和的对象”弄明白，再写 DP。暴力枚举只在 T 很小时能跑，所以它是验证器，不是训练器。

In [ ]:
def ctc_forward_brute_force(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """暴力枚举版 CTC（仅用于验证，T 较小时可用）。
    枚举所有长度为 T 的路径，保留折叠后等于 labels 的路径，用 logsumexp 聚合。
    """
    from itertools import product
    T, V = log_probs.shape

    def collapse(path):
        """去掉 blank、合并重复字符 → 等价于 CTC collapse。"""
        result, prev = [], None
        for c in path:
            if c == blank:
                prev = None
            elif c != prev:
                result.append(c)
                prev = c
        return result

    log_p_list = []
    for path in product(range(V), repeat=T):
        if collapse(list(path)) == list(labels):
            lp = sum(log_probs[t, path[t]] for t in range(T))
            log_p_list.append(lp)

    if not log_p_list:
        return -1e30

    arr = np.array(log_p_list)
    m = arr.max()
    return float(m + np.log(np.exp(arr - m).sum()))


# 快速自检（T=4 时暴力枚举可在 < 1 秒内完成）
np.random.seed(42)
_lp4 = log_softmax(np.random.randn(4, VOCAB_SIZE))
_ref = ctc_forward_brute_force(_lp4, LABELS, blank=BLANK)
print(f"暴力枚举参考值（T=4）: {_ref:.4f}  → 用于断言 DP 实现正确性")

In [ ]:
try:
    lp = ctc_forward(log_probs, LABELS, blank=BLANK)
    print(f'log P(labels | frames) = {lp:.4f}')
    print(f'P(labels | frames)     = {np.exp(lp):.8f}')
    assert np.isfinite(lp), '前向算法应返回有限值'
    assert lp < 0, 'log 概率应为负数'

    # 强验证：与暴力枚举参考实现对照（T=6 时 3^6=729 路径，可在 < 1 秒内枚举）
    lp_ref = ctc_forward_brute_force(log_probs, LABELS, blank=BLANK)
    print(f'暴力枚举参考值         = {lp_ref:.4f}')
    assert np.isclose(lp, lp_ref, atol=1e-5), \
        f'❌ 与参考值不符：DP={lp:.4f}, brute={lp_ref:.4f}，请检查递推或初始条件'

    print('✅ CTC 前向算法验证通过（与暴力枚举吻合至 1e-5）')
except NotImplementedError:
    print('⚠️ ctc_forward 尚未实现，请完成 TODO 步骤1-3')

## 复杂度与直觉

In [ ]:
try:
    # 展示：T 增大时计算量线性增长（而不是指数增长）
    print(f'{"T":>4}  {"S=2L+1":>6}  {"计算格数":>8}  {"log P":>8}')
    print('-' * 35)
    for T_test in [5, 10, 20, 50, 100]:
        _logits_t = np.random.randn(T_test, VOCAB_SIZE)
        _logits_t -= _logits_t.max(axis=1, keepdims=True)  # 数值稳定
        lp_test = _logits_t - np.log(np.exp(_logits_t).sum(axis=1, keepdims=True))
        result = ctc_forward(lp_test, LABELS, blank=BLANK)
        S = 2*len(LABELS)+1
        print(f'{T_test:>4}  {S:>6}  {T_test*S:>8}  {result:>8.3f}')
except NotImplementedError:
    print('⚠️ ctc_forward 尚未实现，请先完成 TODO 步骤1-3')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| 扩展标签 l' | blank 插在每个标签之间，长度 2L+1 |
| 前向变量 α[t][s] | log 域递推，避免下溢 |
| 三个前驱 | 停留、跳 1、跳 2（限字符不同时）|
| 复杂度 | O(T·S) — 其中 `S = 2L + 1`，与暴力枚举路径的指数复杂度相比快得多 |
| L70 | Whisper 用 Attention 解码，但仍然是 token-by-token 的自回归过程 |

下一课（L70）从代码层面解剖 Whisper 架构：音频编码器（encoder）、交叉注意力（cross-attention）和文本解码器的完整实现。先把本课的 log 域 DP 和 L67 的填表思维串起来，再去看 Whisper。

## ✏️ 闭卷推导检查格 — CTC 前向算法 α 递推

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目 1 — α 递推公式**：CTC 前向变量 $\alpha(t, s)$ 满足递推：

$$\alpha(t, s) = \Big[\alpha(t-1, s) + \alpha(t-1, s-1) + \alpha(t-1, s-2) \cdot \mathbf{1}[\text{条件}]\Big] \cdot p(l'_s \mid x_t)$$

写出 $\mathbf{1}[\text{条件}]$ 的完整条件（涉及 blank 与相邻非 blank 标签的关系）。

**题目 2 — blank 合并规则**：

| 路径 | 折叠后标签 |
|------|-----------| 
| a - a | |
| a - ε - a | |
| a - ε - b | |

（ε = blank 符号）

**题目 3 — 扩展标签**：标签序列 $l = [a, b]$，写出 $l'$（含 blank）= ?

（在此处写推导...）

In [ ]:
# 验证：与预计算参考值做数值一致性对比（固定 seed=42）
import numpy as np

# 小规模例子：vocab={blank=0, a=1, b=2}，标签 l=[a,b]
# 扩展标签 l' = [ε, a, ε, b, ε]
T, vocab_size = 5, 3
np.random.seed(42)
log_probs_raw = np.random.dirichlet(np.ones(vocab_size), size=T)
log_probs = np.log(log_probs_raw + 1e-9)  # (T, vocab)

l_prime = [0, 1, 0, 2, 0]  # [blank, a, blank, b, blank]
S = len(l_prime)

# 前向递推（参考实现，不可修改）
NEG_INF = float('-inf')
log_alpha = np.full((T, S), NEG_INF)
log_alpha[0][0] = log_probs[0][l_prime[0]]
if S > 1: log_alpha[0][1] = log_probs[0][l_prime[1]]

for t in range(1, T):
    for s in range(S):
        paths = [log_alpha[t-1][s]]
        if s > 0:
            paths.append(log_alpha[t-1][s-1])
        # 第三路径：只在当前不是 blank 且与前一个非 blank 标签不同时才合法
        if s > 1 and l_prime[s] != 0 and l_prime[s] != l_prime[s-2]:
            paths.append(log_alpha[t-1][s-2])
        log_alpha[t][s] = np.logaddexp.reduce(paths) + log_probs[t][l_prime[s]]

# 最终 log P("ab" | 输入) = logaddexp(α(T-1, S-1), α(T-1, S-2))
log_p = np.logaddexp(log_alpha[-1][-1], log_alpha[-1][-2])

# 预计算参考值（seed=42 下固定）
REF_LOG_P = -1.361933   # 通过参考实现预先计算

assert not np.isnan(log_p), "log_p 是 NaN，递推可能有数值问题"
assert log_p < 0, "log 概率必须 < 0"
assert abs(log_p - REF_LOG_P) < 1e-4, (
    f"log_p = {log_p:.6f}，参考值 = {REF_LOG_P:.6f}\n"
    f"差值 = {abs(log_p - REF_LOG_P):.2e}（应 < 1e-4）\n"
    f"请检查题目 1 中 α 递推的三路径条件是否正确"
)
print(f"log P(ab | 输入) = {log_p:.6f}  参考值 = {REF_LOG_P:.6f}")
print("✅ CTC 前向算法验证通过（数值与参考一致）")

In [ ]:
# ── 手工验证：2帧 · 3词表（含blank=0）─────────────────────────────────
# 设定：vocab = {0:blank, 1:'a', 2:'b'}，标签序列 = [1]（即 'a'）
# 帧数 T=2，用极简概率矩阵手算前向概率
import numpy as np

# log 概率矩阵 shape=(T, V) — 用对称简单值便于手算
log_probs = np.log(np.array([
    [0.5, 0.4, 0.1],   # t=0: blank=0.5, a=0.4, b=0.1
    [0.3, 0.6, 0.1],   # t=1: blank=0.3, a=0.6, b=0.1
]))
labels = [1]   # target = 'a'

# 手算期望值（blank=0 默认）：
# 扩展标签 l' = [0(blank), 1(a), 0(blank)]
# 有效路径（折叠后等于 [a]）：
#   blank→a (0,1): 0.5×0.6 = 0.30
#   a→blank (1,0): 0.4×0.3 = 0.12
#   a→a     (1,1): 0.4×0.6 = 0.24  ← 连续相同字符折叠合并，合法路径
# P(labels|probs) = 0.30 + 0.12 + 0.24 = 0.66
expected_log_p = np.log(0.66)

# 调用本课实现的 ctc_forward（blank 默认值 0，与本例一致）
log_p = ctc_forward(log_probs, labels)   # blank=0 (default)

np.testing.assert_allclose(log_p, expected_log_p, atol=0.05,
    err_msg=f"CTC 前向概率与手算值不符：得到 {np.exp(log_p):.4f}，期望 ≈ 0.66")
print(f"✅ CTC 手工验证通过：P(a|frame) = {np.exp(log_p):.4f}（期望 ≈ 0.66）")
print("   有效路径：blank→a(0.30)，a→blank(0.12)，a→a(0.24)（连续同字符可折叠）")

In [ ]:
# ✏️ 本课自评
l69_review = {
    "exp_path_count_understood": None,  # 理解朴素枚举为何是指数级（V^T条路径）？True/False
    "expanded_label_l_prime":    None,  # 记住l'=[∅,a,∅,b,∅]，长度S=2L+1？True/False
    "forward_var_recursion":     None,  # 能默写α[t][s]的递推公式（情况1/2/初始化）？True/False
    "ctc_forward_impl":          None,  # ctc_forward()实现正确，与暴力枚举误差<1e-5？True/False
    "whiteboard_passed":         None,  # 白板挑战α递推推导通关？True/False
}

unfilled = [k for k, v in l69_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l69_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L69 全部通关！进入 L70：Whisper 架构解析')

---

→ **下一课**　[L70 · Whisper 架构解析](L70_whisper_arch.ipynb)

> 下节课将学习 **Whisper 架构解析**：Log-Mel 输入、Transformer Encoder-Decoder、多任务头。先看本课附录与 L67 复习桥，再去读 Whisper。